# 🔬 Diagnostic Cybersécurité — Analyse Quasicristalline du Système

> *Les quasicristaux de Penrose maîtrisent tous les états de la matière par leur ordre apériodique parfait — un système de sécurité doit présenter cette même cohérence φ-harmonique. Toute rupture dans l'ordre doré signale une anomalie.*

Ce notebook est un **outil de diagnostic opérationnel** qui :

1. **Scanne le système en temps réel** via le pipeline Sentinel (5 couches)
2. **Analyse le code source** avec les métriques φ-complexity
3. **Croise les signaux** : corrélation bayésienne OS + commits + télémétrie
4. **Détecte les anomalies** par les principes quasicristallins (pavages de Penrose)
5. **Exporte les IoC** au format STIX-like pour la communauté open source

**Prérequis** : `pip install phi-complexity[notebooks]`

---

### Architecture Sentinel — 5 Couches

```
Couche 1 : HostCollector       → Événements système (processus, réseau, fichiers)
Couche 2 : TelemetryNormalizer → Traces normalisées + criticité + enrichissement
Couche 3 : BehaviorAnalyzer    → Patterns MITRE ATT&CK (persistance, C2, exfiltration…)
Couche 4 : BayesianCorrelator  → Score de menace unifié (fusion log-odds pondérée)
Couche 5 : SentinelResponse    → Alertes, IoC, politique de réponse
```

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401 — used in visualization cells below

from phi_complexity.core import PHI, HBAR_PHI
from phi_complexity.notebook_helpers import (
    charger_metriques,
    diagnostic_systeme,
    radar_menaces,
    carte_entropie_penrose,
    carte_heisenberg,
    radar_radiance,
    spirale_doree,
    tableau_diagnostic,
)
from phi_complexity.sentinel import (
    HostCollector, EventType,
    TelemetryNormalizer,
    BehaviorAnalyzer, TypeBehavior, SignalComportemental,
    BayesianCorrelator,
    SentinelResponse,
)
from phi_complexity.commit_risk import (
    FeaturesCommit, scorer_commit,
)

"✦ Modules chargés — Pipeline Sentinel opérationnel"
print(f"  φ = {PHI:.10f}")
print(f"  ħ_φ = {HBAR_PHI:.10f}")
print(f"  Plancher Heisenberg = {HBAR_PHI / 2:.10f}")

## 1. Diagnostic Système Complet (One-Shot)

Le diagnostic intégral exécute les 5 couches Sentinel en une seule commande et croise les résultats avec l'analyse φ-complexity du code source.

> **Principe quasicristallin** : Un système sain est comme un pavage de Penrose — il présente un ordre apériodique φ-cohérent. Toute rupture dans la symétrie quintuple signale une anomalie dissimulée.

In [ ]:
# ── Diagnostic complet du système + code source ──
diag = diagnostic_systeme(cible_code="../phi_complexity")

# Rapport formaté
print(tableau_diagnostic(diag))

# Résumé quantitatif
print(f"\n  Events collectés : {len(diag['events'])}")
print(f"  Traces normalisées : {len(diag['traces'])}")
print(f"  Signaux MITRE : {len(diag['signaux'])}")
print(f"  Alertes : {len(diag['alertes'])}")
print(f"  Fichiers analysés : {len(diag['metriques'])}")

## 2. Analyse des Couches Sentinel — Décomposition

### 2.1 Couche 1 : Collecte des Événements Système

Le `HostCollector` lit directement `/proc` (Linux) ou utilise `ps`/`ss` (fallback) pour collecter :
- **Processus** : PID, nom, ligne de commande, mémoire RSS
- **Réseau** : connexions TCP/UDP, ports, états

In [ ]:
# ── Couche 1 : Collecte brute ──
collector = HostCollector()
events = collector.collecter_tout()
resume = collector.resume()

print(f"Méthode de collecte : {resume['methode']}")
print(f"Processus détectés  : {resume['processus']}")
print(f"Connexions réseau   : {resume['reseau']}")
print(f"Total événements    : {resume['total']}")

# Échantillon de processus
print("\nÉchantillon de processus :")
proc_events = [e for e in events if e.type == EventType.PROCESSUS][:10]
for e in proc_events:
    pid = e.metadata.get("pid", "?")
    nom = e.metadata.get("nom", "?")
    mem = e.metadata.get("vmrss_kb", "?")
    print(f"  PID {pid:>6}  {nom:20s}  RSS={mem} kB")

### 2.2 Couche 2 : Normalisation et Enrichissement

La `TelemetryNormalizer` classifie chaque événement par criticité :
- **INFO** : comportement normal
- **ATTENTION** : légèrement inhabituel (port éphémère en écoute)
- **SUSPECT** : correspond à un pattern connu (processus offensif, encodage base64)
- **CRITIQUE** : menace quasi-certaine (destruction système, backdoor)

In [ ]:
# ── Couche 2 : Normalisation ──
normalizer = TelemetryNormalizer()
traces = normalizer.normaliser(events)
stats = normalizer.statistiques(traces)

print("Statistiques de télémétrie :")
for niveau, count in stats.items():
    if niveau != "total":
        barre = "█" * min(50, count)
        print(f"  {niveau.upper():12s} : {count:4d}  {barre}")
print(f"  {'TOTAL':12s} : {stats['total']:4d}")

# Traces suspectes
suspectes = normalizer.traces_suspectes(traces)
if suspectes:
    print(f"\n⚠  {len(suspectes)} traces suspectes détectées :")
    for t in suspectes[:5]:
        print(f"  [{t.criticite.value.upper()}] {t.evenement.description}")
        print(f"    Tags: {', '.join(t.tags)}")
else:
    print("\n✦ Aucune trace suspecte — télémétrie nominale.")

### 2.3 Couche 3 : Patterns Comportementaux (MITRE ATT&CK)

Le `BehaviorAnalyzer` regroupe les traces par type de menace et calcule une confiance consolidée par fusion bayésienne :

$$\text{confiance}_{\text{consolidée}} = 1 - \prod_i (1 - p_i)$$

Les types couverts (inspirés du framework MITRE ATT&CK) :
- **T1548** : Elevation de privilèges (setuid, chmod +s)
- **T1059** : Command & Scripting (curl|bash, wget|sh)
- **T1071** : Application Layer Protocol (C2)
- **T1027** : Obfuscated Files (base64)
- **T1046** : Network Service Discovery (nmap)

In [ ]:
# ── Couche 3 : Analyse comportementale ──
analyzer = BehaviorAnalyzer()
signaux = analyzer.analyser(traces)

print(f"Signaux comportementaux détectés : {len(signaux)}")
print(f"Score global de menace OS : {analyzer.score_global(signaux) * 100:.1f}%")
print()

if signaux:
    for signal in signaux:
        conf_pct = signal.confiance * 100
        mitre = f" [{signal.mitre_technique}]" if signal.mitre_technique else ""
        print(f"  {signal.type.value.upper():20s}  {conf_pct:5.1f}%{mitre}")
        print(f"    {signal.description}")
        print()
else:
    print("  ✦ Aucun signal — pas de comportement malveillant détecté.")

### 2.4 Radar MITRE ATT&CK — Visualisation

Le radar projette les 10 catégories de menace sur un espace polaire.
Les seuils visuels (40% = vert, 70% = orange) correspondent aux paliers bayésiens du corrélateur.

In [ ]:
# ── Radar MITRE ATT&CK ──
# Simulation avec des signaux réalistes pour démonstration
demo_signaux = [
    SignalComportemental(
        type=TypeBehavior.C2, confiance=0.65,
        description="Connexion réseau sur port C2 connu",
        traces_source=["port:4444"], mitre_technique="T1071",
    ),
    SignalComportemental(
        type=TypeBehavior.PERSISTANCE, confiance=0.85,
        description="curl | bash détecté",
        traces_source=["pid:1234"], mitre_technique="T1059.004",
    ),
    SignalComportemental(
        type=TypeBehavior.DEFENCE_EVASION, confiance=0.55,
        description="Encodage base64 dans la ligne de commande",
        traces_source=["pid:5678"], mitre_technique="T1027",
    ),
    SignalComportemental(
        type=TypeBehavior.RECONNAISSANCE, confiance=0.40,
        description="nmap détecté",
        traces_source=["pid:9012"], mitre_technique="T1046",
    ),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7),
                                subplot_kw={"projection": "polar"})

# Radar avec signaux simulés (menaces)
radar_menaces(demo_signaux, ax=ax1)
ax1.set_title("Simulation — Menaces Actives", fontsize=11, pad=20)

# Radar avec signaux réels du système
radar_menaces(signaux, ax=ax2)
ax2.set_title("Système Courant — État Réel", fontsize=11, pad=20)

plt.tight_layout()
plt.show()

## 3. Corrélation Bayésienne Multi-Signaux (Couche 4)

Le `BayesianCorrelator` fusionne les 3 sources de signaux par mise à jour bayésienne séquentielle dans l'espace des log-odds :

$$\log \frac{P(\text{menace}|X)}{1 - P(\text{menace}|X)} = \log \frac{\pi}{1-\pi} + \sum_k \log \text{LR}_k$$

Chaque signal est pondéré par sa **fiabilité** estimée :
- OS behavior : **0.8** (observation directe de /proc)
- Commit risk : **0.6** (heuristique sur les diffs git)
- Télémétrie : **0.7** (statistiques agrégées)

Le score effectif intègre la fiabilité : $s_{\text{eff}} = r \cdot s + (1-r) \cdot 0.5$

In [ ]:
# ── Couche 4 : Corrélation bayésienne ──
correlator = BayesianCorrelator()

# Scénarios de fusion
scenarios = {
    "Nominal (rien)": ([], [], 0.0),
    "Commit suspect seul": ([], [], 0.6),
    "Signaux OS forts": (demo_signaux, [], 0.0),
    "Commit + OS + Télémétrie": (demo_signaux, traces[:20], 0.45),
}

print("Score de fusion bayésienne par scénario :")
print(f"  {'Scénario':<30s}  {'Score':>6s}  {'Niveau':<10s}")
print("  " + "─" * 50)

scores_scenario = {}
for nom, (sig, trc, sc_commit) in scenarios.items():
    result = correlator.calculer_score(
        signaux=sig, traces=trc, score_commit=sc_commit,
    )
    scores_scenario[nom] = result
    symbole = {"FAIBLE": "✅", "MODÉRÉ": "⚠️", "ÉLEVÉ": "🔴", "CRITIQUE": "🚨"}.get(
        result.niveau, "❓"
    )
    print(f"  {nom:<30s}  {result.score_final * 100:5.1f}%  {symbole} {result.niveau}")

## 4. Carte d'Entropie Penrose — Détection d'Anomalies

> **Principe fondateur** : Les quasicristaux de Penrose exhibent un ordre **apériodique** — jamais répétitif, mais toujours cohérent par le nombre d'or. Le code sain doit présenter cette même signature φ-cohérente.

La carte ci-dessous projette chaque fichier Python dans l'espace (Radiance × Entropie Fibonacci).
- **Zone verte** (Radiance > 85, H_F < H_max) : code hermétique, φ-harmonieux
- **Zone orange** (60 < Radiance < 85) : code en éveil, des zones d'entropie à surveiller
- **Zone rouge** (Radiance < 60, H_F élevée) : complexité dissimulée, risque de backdoor

Les lignes dorées forment la grille de Penrose (angle 36° = π/5), rappelant la symétrie quintuple des quasicristaux.

In [ ]:
# ── Carte d'entropie Penrose ──
metriques = charger_metriques("../phi_complexity")

fig, ax = plt.subplots(figsize=(12, 9))
carte_entropie_penrose(metriques, ax=ax)
plt.tight_layout()
plt.show()

# Statistiques résumées
print(f"\nFichiers analysés : {len(metriques)}")
radiances = [m.get('radiance', 0) for m in metriques]
if radiances:
    print(f"Radiance moyenne   : {sum(radiances)/len(radiances):.1f}")
    print(f"Radiance min       : {min(radiances):.1f}")
    print(f"Radiance max       : {max(radiances):.1f}")
    # Fichiers en zone de risque
    risque = [m for m in metriques if m.get('radiance', 100) < 60]
    if risque:
        print(f"\n⚠  {len(risque)} fichier(s) en zone DORMANT (Radiance < 60) :")
        for m in risque:
            print(f"  ░  {m.get('fichier', '?')} — R={m.get('radiance', 0):.1f}")
    else:
        print("\n✦  Tous les fichiers sont en zone harmonieuse (Radiance ≥ 60).")

## 5. Analyse de Risque des Commits (Phase B)

Le modèle bayésien de `commit_risk` analyse chaque commit git pour détecter :
- **Diffs volumineux** (> 500 lignes)
- **Chemins sensibles** (.github/, *.key, secrets, Dockerfile…)
- **Mots-clés suspects** (bypass, disable, force push, eval()…)
- **Fichiers binaires** modifiés
- **Horodatage** inhabituel (nuit, weekend)

In [ ]:
# ── Simulation de commits à risque ──
commits_demo = [
    FeaturesCommit(sha="abc123", message="feat: add user auth",
                   lignes_ajoutees=50, fichiers_changes=3),
    FeaturesCommit(sha="def456", message="force push --force bypass CI",
                   lignes_ajoutees=800, lignes_supprimees=200,
                   fichiers_changes=25, chemins_sensibles=4,
                   mots_suspects_count=3, hors_heures_bureau=True),
    FeaturesCommit(sha="ghi789", message="fix: remove check disable auth",
                   lignes_ajoutees=10, fichiers_changes=2,
                   chemins_sensibles=1, mots_suspects_count=2),
    FeaturesCommit(sha="jkl012", message="docs: update README",
                   lignes_ajoutees=5, fichiers_changes=1),
    FeaturesCommit(sha="mno345", message="chore: update deps",
                   lignes_ajoutees=100, fichiers_changes=3,
                   chemins_sensibles=2, fichiers_binaires=1,
                   est_weekend=True),
]

print(f"{'SHA':<10s} {'Message':<35s} {'Score':>6s} {'Niveau':<10s}")
print("─" * 65)

scores_commits = []
for feat in commits_demo:
    score, details = scorer_commit(feat)
    niveau = ("FAIBLE" if score < 0.30 else "MODÉRÉ" if score < 0.60
              else "ÉLEVÉ" if score < 0.80 else "CRITIQUE")
    symbole = {"FAIBLE": "✅", "MODÉRÉ": "⚠️",
               "ÉLEVÉ": "🔴", "CRITIQUE": "🚨"}[niveau]
    scores_commits.append((feat.message[:35], score, niveau))
    print(f"{feat.sha:<10s} {feat.message[:35]:<35s} "
          f"{score*100:5.1f}% {symbole} {niveau}")

# Visualisation
fig, ax = plt.subplots(figsize=(10, 5))
messages = [s[0] for s in scores_commits]
scores_vals = [s[1] * 100 for s in scores_commits]
couleurs = [
    "#2ECC40" if s < 30 else "#FF851B" if s < 60
    else "#FF4136" if s < 80 else "#85144b"
    for s in scores_vals
]

bars = ax.barh(messages, scores_vals, color=couleurs,
               edgecolor="black", linewidth=0.5)
ax.axvline(x=30, color="green", linestyle="--", alpha=0.5, label="Seuil FAIBLE")
ax.axvline(x=60, color="orange", linestyle="--", alpha=0.5, label="Seuil MODÉRÉ")
ax.axvline(x=80, color="red", linestyle="--", alpha=0.5, label="Seuil ÉLEVÉ")
ax.set_xlabel("Score de risque (%)")
ax.set_title("Analyse Bayésienne des Commits — Risque de Sécurité")
ax.legend(fontsize=9)
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

## 6. Export IoC et Politique de Réponse

La Couche 5 (`SentinelResponse`) génère des **Indicateurs de Compromission (IoC)** au format STIX-like simplifié, exportables vers les bases de données de threat intelligence communautaires.

C'est ce qui fait la puissance de l'approche : chaque anomalie détectée peut être **partagée** avec la communauté open source, alimentant un cycle vertueux de protection collective.

In [ ]:
# ── Export IoC et politique de réponse ──
responder = SentinelResponse()

# Générer les alertes depuis les signaux de démo
score_demo = correlator.calculer_score(
    signaux=demo_signaux, traces=[], score_commit=0.45,
)
alertes_demo = responder.generer_alertes(score_demo, demo_signaux)

# Rapport Markdown (pour PR/Issues GitHub)
md_rapport = responder.rapport_markdown(
    alertes_demo, titre="Diagnostic Cybersécurité φ"
)
print(md_rapport)

# Export IoC JSON
ioc_json = responder.exporter_ioc_json(alertes_demo)
print("\n── IoC Bundle (STIX-like) ──")
print(ioc_json[:500] + "..." if len(ioc_json) > 500 else ioc_json)

# Politique de réponse
politique = responder.politique_de_reponse(alertes_demo)
print("\n── Politique de Réponse ──")
print(f"  Bloquer PR  : {'OUI 🚫' if politique['bloquer_pr'] else 'NON'}")
print(f"  Escalader    : {'OUI ⬆️' if politique['escalader'] else 'NON'}")
print(f"  Isoler       : {'OUI 🔒' if politique['isoler'] else 'NON'}")
print(f"  Notifier     : {'OUI 📧' if politique['notifier'] else 'NON'}")

## 7. Heisenberg-Phi × Sécurité — Complexité Dissimulée

> *Comme en mécanique quantique, il existe un compromis irréductible :*
> $$\Delta C \cdot \Delta L \geq \frac{\hbar_\varphi}{2} \approx 0.309$$
> *Un code qui dissimule sa complexité (ΔC faible) paiera en lisibilité (ΔL élevé) — et inversement.*

En cybersécurité, ce principe est crucial : un malware bien dissimulé aura un ΔL anormalement élevé (entropie de lisibilité forte) pour un ΔC artificiellement bas (complexité apparemment faible).

La carte Heisenberg-Phi ci-dessous identifie ces anomalies.

In [ ]:
# ── Carte Heisenberg-Phi — Détection de dissimulation ──
if metriques:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

    # Carte Heisenberg
    carte_heisenberg(metriques, ax=ax1)
    ax1.set_title("Carte Heisenberg-Phi\n(ΔC · ΔL ≥ ħ_φ/2)", fontsize=11)

    # Distribution des tensions quantiques
    tensions = [m.get("heisenberg_tension", 0) for m in metriques]
    ax2.hist(tensions, bins=20, color="#FFD700", edgecolor="black", alpha=0.8)
    ax2.axvline(x=1.0, color="red", linestyle="--", linewidth=2,
                label="Seuil quantique (τ=1)")
    ax2.set_xlabel("Tension Quantique (τ = ΔC·ΔL / plancher)")
    ax2.set_ylabel("Nombre de fichiers")
    ax2.set_title("Distribution des Tensions Heisenberg-Phi")
    ax2.legend()

    plt.tight_layout()
    plt.show()

    # Fichiers avec tension anormale
    suspects_heis = [
        (m.get("fichier", "?"), m.get("heisenberg_tension", 0))
        for m in metriques if m.get("heisenberg_tension", 0) > 1.5
    ]
    if suspects_heis:
        print(f"\n⚠  {len(suspects_heis)} fichier(s) avec tension "
              f"Heisenberg > 1.5 :")
        for nom, tension in sorted(suspects_heis, key=lambda x: -x[1]):
            print(f"  🔍  {nom} — τ = {tension:.3f}")
    else:
        print("\n✦  Toutes les tensions sont dans la zone "
              "cohérente (τ ≤ 1.5).")
else:
    print("Aucune métrique chargée.")

## 8. Spirale de Fibonacci et Cohérence Bayésienne

La cohérence bayésienne dorée $C_{\text{Bayes}}$ mesure si les rapports de complexité entre fonctions consécutives suivent la progression φ :

$$C_{\text{Bayes}} = \frac{1}{n-1} \sum_{i=1}^{n-1} \left| \frac{\kappa_{i+1}}{\kappa_i} - \varphi \right|$$

$C_{\text{Bayes}} = 0$ signifie que chaque paire de fonctions suit exactement le ratio doré — le code est **cristallin**.
$C_{\text{Bayes}} \gg 0$ signifie une distribution chaotique — le code est **amorphe**.

In [ ]:
# ── Spirale dorée + cohérence bayésienne ──
if metriques:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Radiance moyenne → spirale
    rad_moy = sum(
        m.get("radiance", 60) for m in metriques
    ) / len(metriques)
    spirale_doree(radiance=rad_moy, ax=axes[0])

    # Distribution C_Bayes
    c_bayes = [m.get("coherence_bayes", 0) for m in metriques]
    axes[1].hist(c_bayes, bins=15, color="#7FDBFF",
                 edgecolor="black", alpha=0.8)
    axes[1].axvline(x=PHI, color="gold", linestyle="--", linewidth=2,
                    label=f"φ = {PHI:.3f}")
    axes[1].set_xlabel("C_Bayes (écart au ratio doré)")
    axes[1].set_ylabel("Nombre de fichiers")
    axes[1].set_title("Distribution de la Cohérence Bayésienne")
    axes[1].legend()

    # Radar du fichier le plus radiant
    meilleur = max(metriques, key=lambda m: m.get("radiance", 0))
    radar_radiance(meilleur, ax=axes[2])

    plt.tight_layout()
    plt.show()
else:
    print("Aucune métrique chargée.")

## 9. Conclusion — La Puissance Dormante

Ce notebook démontre que phi-complexity transforme Jupyter en un **outil de diagnostic cybersécurité opérationnel** :

| Capacité | Module | Principes |
|---|---|---|
| Scan système temps réel | `sentinel.host` | Lecture directe /proc, zéro dépendance |
| Détection MITRE ATT&CK | `sentinel.behavior` | Fusion bayésienne des confiances |
| Corrélation multi-signaux | `sentinel.bayesian` | Log-odds pondérés + fiabilité |
| Analyse de commits | `commit_risk` | Bayésien naïf sur features git |
| Détection de dissimulation | `metriques` (Heisenberg-Phi) | ΔC · ΔL ≥ ħ_φ/2 |
| Carte d'anomalies | Pavages de Penrose | Symétrie quintuple + apériodicité φ |
| Export communautaire | `sentinel.response` | IoC au format STIX-like |

> *Le poids est dissimulable en données — mais pas dans l'espace φ.
> Les invariants du nombre d'or détectent ce que les méthodes classiques ne voient pas :
> la rupture de l'harmonie quasicristalline.*

**Prochaines étapes** :
- Intégration CI/CD avec blocage automatique des PRs à risque
- Dashboard temps réel via Jupyter Widgets
- Corrélation avec les bases CVE/NVD
- Export STIX 2.1 complet pour interopérabilité MISP/OpenCTI

---
*Ancré dans le Morphic Phi Framework — φ-Meta 2026*